# # The Generator Secret: Crushing Large Log Files Without Crashing

In this Colab notebook, we'll harness the power of Python generators to process massive log files without ever running out of memory. We'll generate a synthetic log, process it efficiently, and count unique daily users. 

**Run the cells in order from top to bottom.**

In [ ]:
import datetime
import random
import time
import os

# Let's start by generating a synthetic log file. We'll make it big enough to feel the pain, but small enough to run quickly here.
# This file will have 10 million lines, each in the format 'YYYY-MM-DD HH:MM:SS,USER_ID'.

log_filename = 'synthetic_large_log.log'
num_lines = 100_000 # Reduced for faster execution within Colab cell limits

def generate_log_file(filename, n_lines):
    print(f'Generating {n_lines} lines to {filename}...')
    # Use a simple date range to ensure some daily variation
    start_date = datetime.date(2023, 1, 1)
    end_date = datetime.date(2023, 1, 10)
    days = (end_date - start_date).days + 1
    
    # We'll use a set to generate unique user IDs to make them somewhat realistic
    user_ids = [f'user_{i:05d}' for i in range(1000)]
    random.shuffle(user_ids)
    
    with open(filename, 'w') as f:
        for i in range(n_lines):
            # Pick a random day within our range
            random_day_offset = random.randint(0, days - 1)
            current_date = start_date + datetime.timedelta(days=random_day_offset)
            
            # Generate a random time within the day
            random_hour = random.randint(0, 23)
            random_minute = random.randint(0, 59)
            random_second = random.randint(0, 59)
            current_time = datetime.time(random_hour, random_minute, random_second)
            
            timestamp = datetime.datetime.combine(current_date, current_time)
            user_id = random.choice(user_ids)
            
            f.write(f'{timestamp.strftime('%Y-%m-%d %H:%M:%S')},{user_id}\n')
            
            if (i + 1) % 100_000 == 0:
                print(f'  {i + 1}/{n_lines} lines written...')
    print(f'Finished generating {filename}.')

# Generate the log file
generate_log_file(log_filename, num_lines)

# Check the file size to confirm it's been created
print(f'Log file {log_filename} created with size: {os.path.getsize(log_filename) / (1024*1024):.2f} MB')

Now, let's process this log file. The naive approach would be to load the entire file into memory, but we know that fails spectacularly with large files. Instead, we'll use a generator to read the file line by line. This ensures our memory footprint stays incredibly small, regardless of the file's size.

In [ ]:
from collections import defaultdict

def read_log_with_generator(filename):
    """A generator that reads a log file line by line."""
    print(f'Starting to read log file: {filename} using a generator...')
    with open(filename, 'r') as f:
        for line in f:
            yield line.strip() # strip() removes leading/trailing whitespace, including newline characters

def parse_date_and_user(line):
    """Parses a log line to extract the date and user ID."""
    try:
        timestamp_str, user_id = line.split(',')
        date_str = timestamp_str.split(' ')[0]
        return date_str, user_id
    except ValueError:
        # Handle malformed lines gracefully, though our synthetic data shouldn't have them
        return None, None

# This is where the magic happens: we iterate over the generator
# and process each line without ever holding the whole file in memory.
# We'll store unique users per day in a dictionary where keys are dates
# and values are sets of user IDs for that day.

print('Processing log file and calculating unique daily users...')
start_time = time.time()

unique_daily_users_data = defaultdict(set)

line_count = 0
for line in read_log_with_generator(log_filename):
    date_str, user_id = parse_date_and_user(line)
    if date_str and user_id:
        unique_daily_users_data[date_str].add(user_id)
    line_count += 1
    if line_count % 1_000_000 == 0:
        print(f'  Processed {line_count} lines...')

end_time = time.time()
processing_time = end_time - start_time

print(f'Finished processing {line_count} lines in {processing_time:.2f} seconds.')

# Now, let's display the results.
# First, the total number of unique users across all days.

total_unique_users_overall = sum(len(users) for users in unique_daily_users_data.values())
print(f'\nTotal unique users across all days: {total_unique_users_overall}')

# And then, the number of unique users for each day.
print('\nUnique users per day:')
for date_str, users in sorted(unique_daily_users_data.items()):
    print(f'  {date_str}: {len(users)} unique users')

As you can see, the processing time is relatively fast for this smaller dataset, and importantly, the memory usage remained minimal. The `defaultdict(set)` efficiently stores unique users per day without exploding in memory. This approach scales to files much larger than what would fit into RAM.

In [ ]:
import pandas as pd

# Let's prepare the data for saving to a CSV file.
# We'll convert our dictionary of sets into a list of dictionaries for easier DataFrame creation.

output_data = []
for date_str, users in unique_daily_users_data.items():
    output_data.append({'date': date_str, 'unique_users': len(users)})

# Create a pandas DataFrame
df_unique_users = pd.DataFrame(output_data)

# Define the output filename
output_csv_filename = 'unique_daily_users.csv'

# Save the DataFrame to a CSV file
df_unique_users.to_csv(output_csv_filename, index=False)

print(f'\nSuccessfully saved unique daily user counts to {output_csv_filename}')
print(f'File path: {os.path.abspath(output_csv_filename)}')

# Display the first few rows of the saved data
print('\nFirst 5 rows of the saved CSV:')
print(df_unique_users.head())

We've just processed a large synthetic log file using generators, keeping memory usage incredibly low, and saved the results. This notebook demonstrates the core principle: process data as a stream, one item at a time, to avoid memory exhaustion. While the *processing* speed for this specific task might still be a bottleneck for truly massive datasets (as hinted at in the video), the memory problem is solved. You can now process gigabytes, even terabytes, without a memory error.